## Analyze the findings

In [2]:
import numpy as np
import plotly.graph_objects as go
import pandas as pd

from src.segment import load_segmented
from src.loader import load_centerline
from src.config import DELTA_COLOR_CLIP_Q
from src.analysis import (comparable_laps, reference_lap, time_delta,
                          corner_windows, corner_times)

In [3]:
data = load_segmented()
grid, laps, zones, apexes, gaps = data["grid"], data["laps"], data["zones"], data["apexes"], data["gaps"]
cl = load_centerline()
length = cl.attrs["length"]

In [4]:
laps_used = comparable_laps(laps)["lap"].tolist()
ref = reference_lap(laps)

In [5]:
comparable_laps(laps)[["lap", "lap_time"]]

,lap,lap_time
0,10,60.796
1,19,61.180
2,9,61.188
3,12,61.400
4,18,61.432
5,8,61.448
6,11,61.464
7,13,61.764
8,21,61.800
9,23,62.100


In [6]:
# take a slow lap
lap = 20
d = time_delta(grid, ref, lap)
windows = corner_windows(grid, zones, laps_used, length)
ct = corner_times(grid, windows, gaps, laps_used, ref, length)

In [7]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=d["s"], y=d["delta"], mode="lines",
                         name=f"lap {lap} vs lap {ref}", line=dict(width=2)))
fig.add_hline(y=0, line=dict(color="grey", dash="dash"))

for _, z in zones[zones["type"] == "corner"].iterrows():
    start, end = z["s_start"], z["s_end"]
    spans = [(start, length), (0.0, end)] if end < start else [(start, end)]   # zone crossing s = 0
    for x0, x1 in spans:
        fig.add_vrect(x0=x0, x1=x1, fillcolor="lightgrey", opacity=0.3, line_width=0, layer="below")
        fig.add_annotation(x=(x0 + x1) / 2, y=1, yref="paper", showarrow=False,
                           text=f"C{int(z['corner'])}", font=dict(size=11))
fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers", name="corner zone",
                         marker=dict(color="lightgrey", size=12, symbol="square")))

fig.update_layout(title=f"Time delta: lap {lap} vs reference lap {ref} (+ = slower)",
                  xaxis_title="distance s (m)", yaxis_title="delta (s)",
                  width=1400, height=500, showlegend=True, legend=dict(orientation="h", y=-0.2))
fig.show()

In [8]:
x = np.interp(d["s"], cl["s"], cl["x"], period=length)
y = np.interp(d["s"], cl["s"], cl["y"], period=length)
lim = np.nanquantile(np.abs(d["loss_rate"]), DELTA_COLOR_CLIP_Q)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=x, y=y, mode="markers", name=f"lap {lap} vs {ref}: red = losing, blue = gaining",
    marker=dict(size=5, color=d["loss_rate"], colorscale="RdBu_r",
                cmin=-lim, cmax=lim, colorbar=dict(title="ms/m")),
    customdata=np.c_[d["s"], d["delta"], d["loss_rate"]],
    hovertemplate="s %{customdata[0]:.1f} m<br>delta %{customdata[1]:.3f} s"
                  "<br>rate %{customdata[2]:.2f} ms/m<extra></extra>"))

main = apexes[apexes["is_main"]]
fig.add_trace(go.Scatter(x=main["x"], y=main["y"], mode="text", name="corner",
                         text=[f"C{int(c)}" for c in main["corner"]], textfont=dict(size=13)))
fig.add_trace(go.Scatter(x=[x[0]], y=[y[0]], mode="markers", name="start/finish",
                         marker=dict(symbol="x", size=14, color="black")))

fig.update_layout(title=f"Where time goes: lap {lap} vs reference lap {ref}",
                  width=1400, height=850, showlegend=True, legend=dict(orientation="h", y=-0.05),
                  xaxis_title="x (m)", yaxis_title="y (m)")
fig.update_yaxes(scaleanchor="x", scaleratio=1)
fig.show()

In [9]:
c = ct[ct["lap"] == lap].sort_values("corner")
loss = c["loss"].fillna(0)
bad = c[~c["valid"]]

fig = go.Figure()
fig.add_trace(go.Bar(x=[f"C{k}" for k in c["corner"]], y=loss, name="time lost (+) / gained (−)",
                     marker_color=np.where(loss > 0, "firebrick", "steelblue")))
fig.add_trace(go.Scatter(x=[f"C{k}" for k in bad["corner"]], y=[0] * len(bad), mode="markers",
                         name="not valid (edge near a NaN run of s)",
                         marker=dict(symbol="x", size=12, color="grey")))
fig.update_layout(title=f"Time per corner: lap {lap} vs reference lap {ref}",
                  xaxis_title="corner", yaxis_title="loss (s)",
                  width=1400, height=500, showlegend=True, legend=dict(orientation="h", y=-0.2))
fig.show()

In [11]:
def track_xy(s_from: float, s_to: float) -> tuple[np.ndarray, np.ndarray]:
    """Centerline x/y from s_from to s_to (driving direction, crosses s = 0 if needed)."""
    span = np.mod(s_to - s_from, length)
    s = np.mod(s_from + np.arange(0, span + 0.5, 0.5), length)
    return (np.interp(s, cl["s"], cl["x"], period=length),
            np.interp(s, cl["s"], cl["y"], period=length))

loss_by_corner = ct[ct["lap"] == lap].set_index("corner")["loss"]

fig = go.Figure()
fig.add_trace(go.Scatter(x=cl["x"], y=cl["y"], mode="lines", name="track",
                         line=dict(color="lightgrey", width=3)))

for i, (_, w) in enumerate(windows.iterrows()):
    x, y = track_xy(w["s_start"], w["s_end"])
    fig.add_trace(go.Scatter(x=x, y=y, mode="lines", name="corner window",
                             legendgroup="win", showlegend=(i == 0),
                             line=dict(color="orange", width=10)))

    z = zones[zones["zone"] == w["zone"]].iloc[0]
    xz, yz = track_xy(z["s_start"], z["s_end"])
    fig.add_trace(go.Scatter(x=xz, y=yz, mode="lines", name="corner zone",
                             legendgroup="zone", showlegend=(i == 0),
                             line=dict(color="black", width=3)))

    loss = loss_by_corner.get(w["corner"], np.nan)
    label = f"C{int(w['corner'])} {loss * 1000:+.0f} ms" if pd.notna(loss) else f"C{int(w['corner'])} n/a"
    fig.add_trace(go.Scatter(x=[x[0]], y=[y[0]], mode="markers+text", name=" corner window start",
                             legendgroup="start", showlegend=(i == 0),
                             marker=dict(symbol="triangle-up", size=11, color="green"),
                             text=[label], textposition="top center", textfont=dict(size=11)))

fig.update_layout(title=f"Corner windows on track (loss: lap {lap} vs reference lap {ref})",
                  width=1400, height=900, showlegend=True, legend=dict(orientation="h", y=-0.05),
                  xaxis_title="x (m)", yaxis_title="y (m)")
fig.update_yaxes(scaleanchor="x", scaleratio=1)
fig.show()

This is not working, takig the earliest brake point across laps is too fragile, any early dab or braking from the previous corner pulls the window far back. 

In [12]:
lap_b = ref       

from src.segment import BRAKE_COL

g = grid[grid["lap"] == lap_b].sort_values("s")
bx = np.interp(g["s"], cl["s"], cl["x"], period=length)
by = np.interp(g["s"], cl["s"], cl["y"], period=length)
pmax = grid.loc[grid["lap"].isin(laps_used), BRAKE_COL].max()     # same scale for every lap

fig = go.Figure()
fig.add_trace(go.Scatter(x=cl["x"], y=cl["y"], mode="lines", name="track",
                         line=dict(color="black", width=1)))
fig.add_trace(go.Scatter(
    x=bx, y=by, mode="markers", name=f"lap {lap_b} front brake pressure",
    marker=dict(size=7, color=g[BRAKE_COL], colorscale=[[0, "white"], [1, "darkred"]],
                cmin=0, cmax=pmax, line=dict(width=0.3, color="grey"),
                colorbar=dict(title="bar")),
    customdata=np.c_[g["s"], g[BRAKE_COL], g["phase"].astype(str)],
    hovertemplate="s %{customdata[0]:.1f} m<br>brake %{customdata[1]:.1f} bar"
                  "<br>phase %{customdata[2]}<extra></extra>"))

main = apexes[apexes["is_main"]]
fig.add_trace(go.Scatter(x=main["x"], y=main["y"], mode="text", name="corner (apex)",
                         text=[f"C{int(c)}" for c in main["corner"]], textfont=dict(size=12)))

wx0 = np.interp(windows["s_start"], cl["s"], cl["x"], period=length)
wy0 = np.interp(windows["s_start"], cl["s"], cl["y"], period=length)
fig.add_trace(go.Scatter(x=wx0, y=wy0, mode="markers", name="current window start",
                         marker=dict(symbol="triangle-up", size=11, color="green"),
                         customdata=windows["corner"],
                         hovertemplate="window start C%{customdata}<extra></extra>"))

fig.update_layout(title=f"Front brake pressure on track, lap {lap_b}",
                  width=1400, height=900, showlegend=True, legend=dict(orientation="h", y=-0.05),
                  xaxis_title="x (m)", yaxis_title="y (m)", plot_bgcolor="whitesmoke")
fig.update_yaxes(scaleanchor="x", scaleratio=1)
fig.show()

in some corners there is no braking at all so let's look at the lifting instead

In [30]:
lap_b = 20
g = grid[grid["lap"] == lap_b].sort_values("s")
gx = np.interp(g["s"], cl["s"], cl["x"], period=length)
gy = np.interp(g["s"], cl["s"], cl["y"], period=length)
s_vals = g["s"].to_numpy()

fig = go.Figure()
fig.add_trace(go.Scatter(x=cl["x"], y=cl["y"], mode="lines", name="track",
                         line=dict(color="lightgrey", width=4)))

for name, phases, color in [("lift", ["lift"], "orange"),
                            ("trail braking", ["trail braking"], "blue"),
                            ("braking", ["braking"], "red") ]:
    m = g["phase"].isin(phases).to_numpy()
    fig.add_trace(go.Scatter(x=gx[m], y=gy[m], mode="markers", name=f"lap {lap_b}: {name}",
                             marker=dict(size=6, color=color),
                             customdata=np.c_[s_vals[m], g["phase"].to_numpy()[m]],
                             hovertemplate="s %{customdata[0]:.1f} m<br>%{customdata[1]}<extra></extra>"))

main = apexes[apexes["is_main"]]
fig.add_trace(go.Scatter(x=main["x"], y=main["y"], mode="text", name="corner (apex)",
                         text=[f"C{int(c)}" for c in main["corner"]], textfont=dict(size=12)))

fig.update_layout(title=f"Lift and braking on track, lap {lap_b}",
                  width=1400, height=900, showlegend=True, legend=dict(orientation="h", y=-0.05),
                  xaxis_title="x (m)", yaxis_title="y (m)")
fig.update_yaxes(scaleanchor="x", scaleratio=1)
fig.show()

Since the behavior is inconsistent with the braking and lifitng around corners, the following could be a strategy:
1. we use the braking zone that is before that corner's apex and after the previous apex, if there are multiple interrupted ones take the longer one's starting point. if the apex of the previous corner happens to be inside the braking zone, we do not consider it.
2. if there is no braking then we take trail braking.
3. if there are no braking or trail braking zones between that corner's apex and the apex of the corner before it, we take the lift section between the two apexes. if there are multiple ones, take the closest one to that corner's apex.
4. if there is no braking, no trail braking, and no lifting, take that corner's zone start s form the grid.
+a flag that includes if there was braking, train braking, lifting, or corner zone.